# 비유의성 해석을 위한 공연 노출 측정 진단
왜 통계적으로 유의하지 않았는지 한 가지 원인을 확정하지 않는다. 여기서는 **이미 관측된 노출의 측정·집계 구조**를 확인하고, 그 구조가 실제 공연 관람객의 소비 신호와 어떻게 달라질 수 있는지 해석한다. 신호 약화가 가능한 경로와 실제 추정치를 약화시켰다고 증명한 결과는 구분한다.

고정 KOPIS 상세, 기존 500m 공연–상권 연결, 현재 폴리곤, 최종 예정회차 및 6개 매출 시간구간 자료만 읽는다. 새로운 회귀·유의성 탐색·네트워크 요청은 하지 않는다. 실제 관객이 없는 상태에서 객석수를 관객으로 대체하지 않는다. 모든 산출물은 `reports/20261006/null_diagnostics/exposure_*`로 저장한다.

## 1. 입력 해시 확인
기존 최종 통계가 사용한 입력 해시 및 09번 노출 구축 메타데이터의 결과 해시를 확인한다. 공간 진단은 고정 서울 경계 명세가 지시하는 투영 폴리곤을 그대로 읽어 계산한다.

In [ ]:
# 현재 구조 및 과거 고정 명세의 경로를 함께 해석합니다.
from pathlib import Path as _ProjectPath
import json as _bootstrap_json
_project_home = next(p for p in [_ProjectPath.cwd(), *_ProjectPath.cwd().parents]
                     if (p / 'environment/project_paths.ipynb').is_file())
_project_notebook = _bootstrap_json.loads((_project_home / 'environment/project_paths.ipynb').read_text(encoding='utf-8'))
for _project_cell in _project_notebook['cells']:
    if 'project-runtime' in _project_cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_project_cell['source']), 'environment/project_paths.ipynb', 'exec'), globals())

from pathlib import Path
import hashlib,json,re
import numpy as np
import pandas as pd
from IPython.display import display,Markdown
ROOT=PROJECT_ROOT
OUT=project_path('reports/20261006/null_diagnostics'); OUT.mkdir(parents=True,exist_ok=True)
inputs={}
def register(relative,expected=None):
    path=project_path(relative); actual=hashlib.sha256(path.read_bytes()).hexdigest()
    if expected is not None: assert actual==expected,f'고정 해시 불일치: {relative}'
    inputs[relative]=dict(path=relative,sha256_before=actual,expected_sha256=expected)
    return path
read_json=lambda relative:project_json_loads(register(relative).read_text(encoding='utf-8'))
def read_csv(relative,expected=None,**kwargs):
    return pd.read_csv(register(relative,expected),float_precision='round_trip',**kwargs)
model_meta=read_json('reports/20261006/time_windows/11_analysis_summary.json')
for relative,expected in model_meta['input_sha256'].items():register(relative,expected)
exposure_meta=read_json('reports/20261006/time_windows/09_exposure_summary.json')
scope=read_json('reports/20261006/08_food_scope_metadata.json')
kopis=read_json('jupyter-notebook/20261006/metadata/kopis_analysis_sources.json')
seoul=read_json('jupyter-notebook/20261006/metadata/seoul_analysis_sources.json')

In [ ]:
for relative,record in exposure_meta['outputs'].items():register(relative,record['sha256'])
expected={row['path']:row['sha256'] for row in scope['source_files']}
key_types={'area_id':str,'mt20id':str,'mt10id':str,'quarter':str,'start_quarter':str}
panel=read_csv('dataset/analysis/final/food_industry_panel.csv',scope['panel_sha256'],dtype=key_types)
near=read_csv('dataset/analysis/final/venue_area_distances_800m.csv',expected['dataset/analysis/final/venue_area_distances_800m.csv'],dtype=key_types)
near=near.loc[near.distance_m.le(500)].copy()
links=read_csv('dataset/analysis/final/performance_facility_quarter.csv',expected['dataset/analysis/final/performance_facility_quarter.csv'],dtype=key_types)
areas=read_csv('dataset/analysis/final/study_areas.csv',expected['dataset/analysis/final/study_areas.csv'],dtype=key_types)
sessions=read_csv('dataset/analysis/final/time_windows/nominal_sessions.csv',dtype=key_types,parse_dates=['nominal_start','estimated_end'])
availability=read_csv('dataset/analysis/final/time_windows/availability.csv',dtype=key_types)
audit=read_csv('reports/20261006/time_windows/09_performance_schedule_audit.csv',dtype=key_types)
sales=read_csv('dataset/analysis/final/time_windows/sales_time_panel.csv',dtype=key_types)
frames={}
for role in ['performance_details','facility_details']:
    spec=kopis['files'][role]
    frames[role]=read_csv(spec['path'],spec['sha256'],dtype=str,keep_default_na=False)
performance_details,facility_details=frames['performance_details'],frames['facility_details']
near=near.loc[near.area_id.isin(areas.area_id)]
shared=links.merge(near,on='mt10id',how='inner',validate='many_to_many')
assert shared.link_method.eq('official_detail').all()
assert shared.mt20id.nunique()==12839
candidate_ids=set(shared.mt20id)

**해석:** 500m 분석에 기여한 후보 12,839개는 공식 시설 ID 연결이다. 현재 좌표 확보와 실제 과거 좌표·회차·관객 확보는 서로 다른 문제다.

## 2. 공연 ID와 예정회차는 무엇이 다른가
기존 지표는 같은 분기에 같은 공연 ID가 여러 번 상연될 수 있어도 한 번 센다. 예정회차 지표는 파싱 가능한 ID에서 요일·시각을 반복하여 만든 시작 횟수다. 두 지표는 단위도 다르고 일정 가용 표본도 다르므로 모든 후보 ID와 가용 ID를 각각 비교한다.

새 노출자료의 시작일이 연구기간 밖인 패딩 회차는 이 절의 분기별 회차 수에서 제외한다. 분석 단위는 151개 상권×12분기인 1,812행이다.

In [ ]:
base=panel[['area_id','quarter','known_performance_count_500m']].drop_duplicates().sort_values(['area_id','quarter'])
assert len(base)==1812
original=shared.groupby(['area_id','quarter']).mt20id.nunique()
check=base.set_index(['area_id','quarter']).known_performance_count_500m
assert np.array_equal(check,original.reindex(check.index,fill_value=0))
eligible_ids=set(audit.loc[audit.exposure_eligible,'mt20id'])
eligible=shared.loc[shared.mt20id.isin(eligible_ids)].groupby(['area_id','quarter']).mt20id.nunique()
study_sessions=sessions.loc[sessions.nominal_start.ge('2023-01-01') & sessions.nominal_start.lt('2026-01-01')].copy()
area_sessions=study_sessions.merge(near[['area_id','mt10id']],on='mt10id',how='inner',validate='many_to_many')
session_counts=area_sessions.groupby(['area_id','start_quarter']).size()
aquarter=base.rename(columns={'known_performance_count_500m':'original_id_count'}).set_index(['area_id','quarter'])
aquarter['eligible_id_count']=eligible.reindex(aquarter.index,fill_value=0)
aquarter['nominal_session_count']=session_counts.reindex(aquarter.index,fill_value=0)
aquarter=aquarter.reset_index().merge(availability[['area_id','quarter','availability_pct']],on=['area_id','quarter'],validate='one_to_one')
assert int(study_sessions.shape[0])==exposure_meta['nominal_sessions_study_start']
display(aquarter[['original_id_count','eligible_id_count','nominal_session_count','availability_pct']].describe())

In [ ]:
correlation_rows=[]
for xname in ['original_id_count','eligible_id_count']:
    for subset_label,part in [('all_1812_area_quarters',aquarter),('availability_ge90',aquarter.loc[aquarter.availability_pct.ge(90)])]:
        x,y=part[xname].astype(float),part.nominal_session_count.astype(float)
        lx,ly=np.log1p(x),np.log1p(y)
        for label,xvalue,yvalue in [('levels_pearson',x,y),('levels_spearman',x.rank(),y.rank()),
            ('log1p_pearson',lx,ly),('within_area_log1p_pearson',lx-lx.groupby(part.area_id).transform('mean'),ly-ly.groupby(part.area_id).transform('mean'))]:
            correlation_rows.append(dict(id_measure=xname,sample=subset_label,comparison=label,n=len(part),correlation=xvalue.corr(yvalue)))
    part=aquarter.sort_values(['area_id','quarter']).copy()
    dx=part.groupby('area_id')[xname].diff();dy=part.groupby('area_id').nominal_session_count.diff()
    good=dx.notna() & dy.notna()
    correlation_rows.append(dict(id_measure=xname,sample='adjacent_quarter_pairs',comparison='first_difference_pearson',n=int(good.sum()),correlation=dx[good].corr(dy[good])))
correlations=pd.DataFrame(correlation_rows)
display(correlations.style.format({'correlation':'{:.4f}'}))

In [ ]:
change=aquarter.sort_values(['area_id','quarter']).copy()
change['id_change']=change.groupby('area_id').original_id_count.diff()
change['session_change']=change.groupby('area_id').nominal_session_count.diff()
change=change.dropna(subset=['id_change','session_change'])
any_change=change.id_change.ne(0)|change.session_change.ne(0)
change_summary={
    'adjacent_pairs':len(change),'pairs_with_either_measure_changed':int(any_change.sum()),
    'opposite_sign_changes':int((change.id_change*change.session_change).lt(0).sum()),
    'same_id_count_but_sessions_changed':int((change.id_change.eq(0)&change.session_change.ne(0)).sum())}
change_summary['opposite_sign_pct_of_all_pairs']=100*change_summary['opposite_sign_changes']/len(change)
change_summary['opposite_sign_pct_of_changed_pairs']=100*change_summary['opposite_sign_changes']/int(any_change.sum())
per_id=study_sessions.groupby('mt20id').size()
intensity_summary={'eligible_ids_with_study_start':len(per_id),'nominal_sessions_study_start':len(study_sessions),
    'median_nominal_sessions_per_id':float(per_id.median()),'mean_nominal_sessions_per_id':float(per_id.mean()),
    'p90_nominal_sessions_per_id':float(per_id.quantile(.9)),'max_nominal_sessions_per_id':int(per_id.max()),
    'top10_id_share_of_nominal_sessions_pct':float(100*per_id.nlargest(10).sum()/per_id.sum())}
display(Markdown(f'**관측된 차이:** 예정회차는 가용 ID당 중앙값 {per_id.median():.0f}회, '
    f'90백분위 {per_id.quantile(.9):.0f}회, 최대 {per_id.max():,}회다. '
    f'인접분기 {len(change):,}쌍 중 두 지표의 증감 부호가 반대인 경우는 {change_summary["opposite_sign_changes"]:,}쌍이다.'))

**가능한 영향:** 공연 ID 수는 공연의 반복 횟수 차이를 놓치고, 예정회차는 실제 상연·취소·관객을 놓친다. 상관이 높아도 두 지표가 같은 개념인 것은 아니다. 낮은 상관·변화 불일치 역시 어느 지표가 실제 관객 노출에 더 정확한지를 입증하지 않는다. 이 상관표는 회귀계수나 원인 검정이 아니다.

## 3. 관객·실제 종료·객석 정보의 구분
공연 상세에 날짜별 실제 시작·종료, 실제 관객·판매 티켓·객석 점유율 필드가 있는지 확인한다. 시설 상세의 `seatscale`은 현재 시설 단위 수용규모이며, 같은 시설에 여러 공연장이 있거나 상연별 좌석 배치가 달라질 수 있다. 이를 예정회차에 곱하여 실제 관객으로 만들지 않는다.

In [ ]:
candidate_details=performance_details.loc[performance_details.mt20id.isin(candidate_ids)].copy()
active_facility_ids=set(shared.mt10id)
active_facilities=facility_details.loc[facility_details.mt10id.isin(active_facility_ids)].copy()
seats=pd.to_numeric(active_facilities.seatscale,errors='coerce')
halls=pd.to_numeric(active_facilities.mt13cnt,errors='coerce')
absent_fields=[field for field in ['actual_start','actual_end','session_id','audience_count','ticket_sales_count','occupancy_rate'] if field not in candidate_details.columns]
assert len(absent_fields)==6
candidate_audit=audit.loc[audit.in_500m_candidate].copy()
measurement_rows=[
    ('candidate_performance_IDs',len(candidate_details),len(candidate_details),'performance_ID'),
    ('schedule_text_nonempty',candidate_details.dtguidance.str.strip().ne('').sum(),len(candidate_details),'performance_ID'),
    ('runtime_text_nonempty',candidate_details.prfruntime.str.strip().ne('').sum(),len(candidate_details),'performance_ID'),
    ('exposure_eligible',candidate_audit.exposure_eligible.sum(),len(candidate_details),'performance_ID'),
    ('own_runtime_overlap_ambiguous',candidate_audit.own_runtime_overlap_ambiguous.sum(),len(candidate_details),'performance_ID'),
    ('HOL_mentioned_normal_weekday_repeated',candidate_audit.has_hol.sum(),len(candidate_details),'performance_ID'),
    ('period_extends_beyond_2025',candidate_audit.period_extends_beyond_2025.sum(),len(candidate_details),'performance_ID'),
    ('performance_hall_ID_mt13id_nonempty',candidate_details.mt13id.str.strip().ne('').sum(),len(candidate_details),'performance_ID'),
    ('current_facility_positive_seat_capacity',seats.gt(0).sum(),len(active_facilities),'facility_ID'),
    ('current_facility_multiple_halls',halls.gt(1).sum(),len(active_facilities),'facility_ID')]
measurement=pd.DataFrame(measurement_rows,columns=['metric','numerator','denominator','unit'])
measurement['pct']=100*measurement.numerator/measurement.denominator
display(measurement.style.format({'pct':'{:.2f}'}))
updated=pd.to_datetime(candidate_details.updatedate,format='mixed',errors='coerce')
registered_end=pd.to_datetime(candidate_details.prfpdto,format='%Y.%m.%d',errors='coerce')
historical_schedule_metadata={
    'snapshot_run_id':kopis.get('run_id','20261004_154020_1c7d3b'),
    'candidate_IDs':len(candidate_details),
    'IDs_with_parsed_update_timestamp':int(updated.notna().sum()),
    'IDs_updated_after_2025':int(updated.ge('2026-01-01').sum()),
    'IDs_updated_after_registered_end_date':int((updated.dt.normalize()>registered_end).sum()),
    'min_update_timestamp':str(updated.min()),'max_update_timestamp':str(updated.max()),
    'update_timestamp_proves_schedule_changed':False,
    'historical_schedule_versions_available':False}
display(pd.DataFrame([historical_schedule_metadata]).T.rename(columns={0:'값'}))


**관측과 한계:** 일정 파싱 실패나 기간 재사용은 실제 측정된 가용률·규칙으로 확인된다. 실제 관객 수와 실제 종료를 검증한 레코드는 현재 입력에 없다. 시설 객석수가 있다고 해도 공연별·회차별 점유율과 체류·지출은 관측되지 않는다. 따라서 대형 매진 공연과 소형·저관객 공연을 적절히 구분하지 못할 가능성은 있지만 그로 인한 감쇠 크기·방향은 현재 자료로 계산할 수 없다.

## 4. 같은 공연·시설의 상권 공유
고정된 500m 연결을 그대로 이용해 같은 시설, 같은 공연 ID, 같은 공연×분기가 몇 개 상권에 공유되는지 센다. 이 중복은 물리적으로 다른 공연이 더 생겼다는 뜻이 아니다. 인접 상권의 노출이 함께 움직일 수 있는 구조다.

현재 입력은 2026년 수집된 단일 상세 스냅샷이다. `updatedate`가 연구 종료나 등록 공연 종료 이후인 경우도 구분해 저장하지만, 이 수정시각만으로 일정이 바뀌었다고 단정하지 않는다. 과거 각 날짜에 유효했던 요일·시각 버전과 임시 변경 이력이 없으므로 동일 일정의 과거 재사용 정확성은 검증되지 않았다. 공휴일별 예외 일정과 취소 이력도 현재 예정회차로 복원되지 않는다.

In [ ]:
shares={
    'active_facility_ID':near.loc[near.mt10id.isin(active_facility_ids)].groupby('mt10id').area_id.nunique(),
    'performance_ID':shared.groupby('mt20id').area_id.nunique(),
    'performance_ID_quarter':shared.groupby(['mt20id','quarter']).area_id.nunique()}
sharing_rows=[]
for label,counts in shares.items():
    sharing_rows.append(dict(unit=label,n_units=len(counts),mean_areas_per_unit=float(counts.mean()),
        median_areas_per_unit=float(counts.median()),p90_areas_per_unit=float(counts.quantile(.9)),
        max_areas_per_unit=int(counts.max()),units_shared_by_at_least_2_areas=int(counts.ge(2).sum()),
        shared_unit_pct=float(100*counts.ge(2).mean()),total_area_unit_relations=int(counts.sum())))
sharing=pd.DataFrame(sharing_rows)
display(sharing.style.format(precision=3))

**가능한 영향:** 같은 공연을 여러 상권에 공유하는 노출은 인접 상권을 독립적인 처치·비처치로 해석하기 어렵게 한다. 상권 군집 추론은 같은 상권 안의 상관을 처리하지만 다른 상권 사이의 공간 상관을 자동으로 제거하지 않는다. 공유 정도만으로 실제 회귀잔차의 공간 상관을 확인했다고 말할 수는 없다.

## 5. 공연장점–폴리곤 최단거리 500m의 공간적 의미
기존 기준은 공연장과 상권 **어느 한 점의 최소거리**다. 상권 전체 또는 모든 음식점이 500m 안에 있다는 기준이 아니다. 고정 폴리곤과 현재 시설좌표를 이용해 중심점 거리, 가장 먼 폴리곤 꼭짓점 거리, 상권 면적 중 반경 500m 원 안의 비중을 계산한다.

원 안의 면적 비중은 토지 면적의 기하학적 기술통계다. 음식점의 분포·매출·인구 비중으로 해석하지 않는다. 원은 64개 분절/사분면으로 근사하며 거리·면적 계산은 EPSG:5181에서 수행한다.

In [ ]:
import shapefile,shapely
from shapely.geometry import shape,Point
from pyproj import CRS,Transformer
area_source=next(item for item in seoul['items'] if item['dataset']=='areas')
for entry in area_source['extracted']:register(entry['path'],entry['sha256'])
shp=project_path(next(entry['path'] for entry in area_source['extracted'] if entry['path'].endswith('.shp')))
assert CRS.from_wkt(shp.with_suffix('.prj').read_text()).to_epsg()==5181
reader=shapefile.Reader(str(shp),encoding=shp.with_suffix('.cpg').read_text().strip())
geometry={}
for item in reader.iterShapeRecords():
    area_id=str(item.record.as_dict()['TRDAR_CD']).removesuffix('.0')
    if area_id not in set(areas.area_id):continue
    g=shape(item.shape.__geo_interface__)
    if not g.is_valid:
        g=shapely.make_valid(g)
        if g.geom_type=='GeometryCollection':g=shapely.union_all([part for part in g.geoms if part.geom_type in ['Polygon','MultiPolygon']])
    assert g.is_valid and not g.is_empty and g.area>0
    geometry[area_id]=g
transformer=Transformer.from_crs('EPSG:4326','EPSG:5181',always_xy=True)
xs,ys=transformer.transform(pd.to_numeric(active_facilities.lo),pd.to_numeric(active_facilities.la))
points=dict(zip(active_facilities.mt10id,[Point(x,y) for x,y in zip(xs,ys)]))

In [ ]:
spatial_rows=[]
for row in near.loc[near.mt10id.isin(active_facility_ids)].itertuples(index=False):
    g,point=geometry[row.area_id],points[row.mt10id]
    minimum=float(g.distance(point))
    assert abs(minimum-row.distance_m)<1e-5
    coordinates=shapely.get_coordinates(g)
    farthest=float(np.hypot(coordinates[:,0]-point.x,coordinates[:,1]-point.y).max())
    covered=float(g.intersection(point.buffer(500,quad_segs=64)).area/g.area)
    spatial_rows.append(dict(area_id=row.area_id,mt10id=row.mt10id,min_distance_m=minimum,
        centroid_distance_m=float(g.centroid.distance(point)),max_vertex_distance_m=farthest,
        polygon_area_km2=float(g.area/1e6),polygon_area_fraction_within_500m=covered))
spatial=pd.DataFrame(spatial_rows)
area_geometry=areas.copy()
area_geometry['polygon_area_km2']=area_geometry.area_id.map(lambda key:geometry[key].area/1e6)
area_geometry['bbox_diagonal_m']=area_geometry.area_id.map(lambda key:float(np.hypot(geometry[key].bounds[2]-geometry[key].bounds[0],geometry[key].bounds[3]-geometry[key].bounds[1])))
area_geometry=area_geometry.merge(spatial.groupby('area_id').agg(active_near_facilities=('mt10id','nunique'),
    median_polygon_fraction_within_500m=('polygon_area_fraction_within_500m','median'),
    max_vertex_distance_to_any_linked_facility_m=('max_vertex_distance_m','max')).reset_index(),on='area_id',how='left',validate='one_to_one')
display(area_geometry.sort_values('polygon_area_km2',ascending=False).head(6).style.format(precision=3))

In [ ]:
spatial_summary={
    'active_facility_area_pairs':len(spatial),
    'pairs_centroid_outside_500m':int(spatial.centroid_distance_m.gt(500).sum()),
    'pct_pairs_centroid_outside_500m':float(100*spatial.centroid_distance_m.gt(500).mean()),
    'pairs_some_polygon_vertices_beyond_1000m':int(spatial.max_vertex_distance_m.gt(1000).sum()),
    'pct_pairs_some_polygon_vertices_beyond_1000m':float(100*spatial.max_vertex_distance_m.gt(1000).mean()),
    'median_polygon_area_fraction_within_500m':float(spatial.polygon_area_fraction_within_500m.median()),
    'pairs_less_than_half_polygon_area_within_500m':int(spatial.polygon_area_fraction_within_500m.lt(.5).sum()),
    'pct_pairs_less_than_half_polygon_area_within_500m':float(100*spatial.polygon_area_fraction_within_500m.lt(.5).mean()),
    'max_vertex_distance_m':float(spatial.max_vertex_distance_m.max()),
    'median_area_km2':float(area_geometry.polygon_area_km2.median()),
    'max_area_km2':float(area_geometry.polygon_area_km2.max())}
display(pd.DataFrame([spatial_summary]).T.rename(columns={0:'값'}))

**가능한 영향:** 상권의 가까운 가장자리 때문에 노출된 것으로 분류되더라도 상권 내 먼 곳의 매출까지 결과변수에 포함될 수 있다. 이는 가까운 음식점만의 변화를 희석할 수 있는 구조지만, 음식점별 좌표·매출이 없으므로 실제 희석률이나 어느 상권의 매출이 잘못 귀속됐는지는 추정하지 않는다. 최장 꼭짓점 거리는 음식점까지의 실제 거리도, 도보거리도 아니다.

## 6. 분기 6구간과 예정 종료 후 3시간의 차이
매출표는 날짜 열 없이 `상권×업종×분기×고정시간구간` 한 행으로 저장되어 있다. 6개 구간 길이는 6·5·3·3·4·3시간이다. 예정 종료 후 3시간 창이 이 구간 경계와 얼마나 맞는지 확인한다. 분기 전체의 해당 시각 매출을 공연이 있었던 날짜만의 매출로 분리하는 것은 불가능하다.

In [ ]:
assert not sales.duplicated(['area_id','industry_id','quarter','time_bin']).any()
assert 'date' not in sales.columns and 'sale_datetime' not in sales.columns
bin_hours=sales[['time_bin','bin_hours']].drop_duplicates().sort_values('time_bin')
end_minutes=study_sessions.estimated_end.dt.hour*60+study_sessions.estimated_end.dt.minute
boundaries=np.array([360,660,840,1020,1260,1440,1800,2100,2280,2460,2700,2880])
starts=end_minutes.to_numpy()
internal_boundaries=((boundaries[None,:]>starts[:,None]) & (boundaries[None,:]<(starts+180)[:,None])).sum(axis=1)
exact=np.isin(starts,[660,840,1260])
window_end=study_sessions.estimated_end+pd.Timedelta(hours=3)
start_q=study_sessions.estimated_end.dt.to_period('Q')
end_q=(window_end-pd.Timedelta(microseconds=1)).dt.to_period('Q')
timing_rows=[
    ('nominal_post03_windows',len(study_sessions),len(study_sessions)),
    ('post03_spans_multiple_fixed_timebins',int((internal_boundaries>0).sum()),len(study_sessions)),
    ('post03_exactly_equals_a_fixed_3h_timebin',int(exact.sum()),len(study_sessions)),
    ('post03_crosses_midnight',int((starts+180>1440).sum()),len(study_sessions)),
    ('post03_crosses_quarter',int(start_q.ne(end_q).sum()),len(study_sessions)),
    ('original_sales_timebin_rows',len(sales),len(sales)),
    ('observed_sales_timebin_rows',int(sales.sales_amount.notna().sum()),len(sales))]
timing=pd.DataFrame(timing_rows,columns=['metric','numerator','denominator'])
timing['pct']=100*timing.numerator/timing.denominator
display(timing.style.format({'pct':'{:.2f}'}))

In [ ]:
calendar_rows=[]
for q in sorted(sales.quarter.unique()):
    period=pd.Period(f'{q[:4]}Q{q[-1]}',freq='Q')
    days=(period.end_time.normalize()-period.start_time.normalize()).days+1
    for row in bin_hours.itertuples(index=False):
        hours=days*row.bin_hours
        calendar_rows.append(dict(quarter=q,time_bin=row.time_bin,days_in_quarter=days,
            bin_hours_per_day=int(row.bin_hours),calendar_hours_in_quarter_bin=hours,
            one_three_hour_window_fraction_of_calendar_hours=3/hours))
calendar=pd.DataFrame(calendar_rows)
display(calendar.groupby('time_bin').agg(bin_hours_per_day=('bin_hours_per_day','first'),
    min_quarter_calendar_hours=('calendar_hours_in_quarter_bin','min'),
    max_quarter_calendar_hours=('calendar_hours_in_quarter_bin','max')))

**가능한 영향:** 같은 종료 후 3시간 노출이 두 구간으로 나뉘기도 하고, 매출 한 행에는 약 90일 동안의 해당 시간대 거래가 모두 포함된다. 공연 날짜의 짧은 효과가 있다면 그 밖의 거래 변동과 섞일 수 있다. 그러나 위의 달력시간 비율을 매출 효과의 감쇠율로 쓰거나 관측 매출을 그 비율로 나눠 공연 후 매출을 만들 수는 없다.

## 7. 근거 저장과 해석 경계
아래 파일은 측정 구조를 설명하는 기술 진단이다. 유의하지 않은 원인을 입증하거나 p값을 낮추기 위한 모형 선택에 사용하지 않는다. 입력 전후 해시와 산출물 해시를 함께 저장한다.

In [ ]:
outputs={
    'exposure_area_quarter_comparison.csv':aquarter,
    'exposure_correlations.csv':correlations,
    'exposure_measurement_availability.csv':measurement,
    'exposure_sharing_summary.csv':sharing,
    'exposure_spatial_pairs.csv':spatial,
    'exposure_area_geometry.csv':area_geometry,
    'exposure_time_alignment.csv':timing,
    'exposure_quarter_timebin_calendar.csv':calendar}
for name,frame in outputs.items():frame.to_csv(OUT/name,index=False,encoding='utf-8-sig')
for relative,record in inputs.items():
    record['sha256_after']=hashlib.sha256((project_path(relative)).read_bytes()).hexdigest()
    record['unchanged']=record['sha256_before']==record['sha256_after']
assert all(row['unchanged'] for row in inputs.values())

In [ ]:
summary={
    'created_utc':pd.Timestamp.now(tz='UTC').isoformat(),
    'role':'descriptive_exposure_measurement_diagnostics_not_causal_attribution_of_nonsignificance',
    'network_requests':0,'new_regression_fits':0,'all_inputs_unchanged':True,
    'id_vs_nominal_session_intensity':intensity_summary,'quarter_change_comparison':change_summary,
    'correlations':correlations.to_dict('records'),'sharing':sharing.to_dict('records'),
    'measurement_availability':measurement.to_dict('records'),'absent_structured_actual_fields':absent_fields,
    'actual_session_end_or_attendance_observed':False,
    'performance_detail_schema':list(candidate_details.columns),
    'historical_schedule_metadata':historical_schedule_metadata,
    'facility_capacity_used_as_audience':False,'schedule_historical_stability_verified':False,
    'spatial':spatial_summary,'time_alignment':timing.to_dict('records'),
    'quarter_timebin_calendar_hours_min':int(calendar.calendar_hours_in_quarter_bin.min()),
    'quarter_timebin_calendar_hours_max':int(calendar.calendar_hours_in_quarter_bin.max()),
    'observation_vs_implication':[
        {'observed':'ID counts and inferred scheduled-session counts use different intensity units and differ in quarterly changes',
         'possible_implication':'event frequency signal may be mismeasured; neither proxy validates actual audience dose'},
        {'observed':'actual endings, attendance, cancellations and historical schedule records are not in supplied structured data',
         'possible_implication':'timing and audience exposure error may attenuate, distort, or add noise; size and direction not identified'},
        {'observed':'performances and facilities are shared by many study areas under point-to-polygon minimum distance',
         'possible_implication':'exposure overlap and spatial dependence may reduce independent contrast; residual dependence not established'},
        {'observed':'some linked polygons extend far beyond the 500m circle and only an area fraction lies nearby',
         'possible_implication':'whole-area sales can mix near and far businesses; actual restaurant or sales weights unobserved'},
        {'observed':'outcomes are quarterly fixed-time-bin sales, not sales on the exact event date and post-event window',
         'possible_implication':'short event effects may mix with many other transactions; no numerical attenuation rate can be computed'}],
    'input_records':list(inputs.values()),
    'outputs':{name:{'rows':len(frame),'sha256':hashlib.sha256((OUT/name).read_bytes()).hexdigest()} for name,frame in outputs.items()}}
(OUT/'exposure_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print(f'완료: CSV {len(outputs)}개 + exposure_summary.json; 입력 {len(inputs)}개 불변')

**최종 판단:** 실제 관객·실제 종료가 관측되지 않고, 공연 ID·예정회차·분기 고정시간 매출의 단위가 다르며, 같은 공연이 여러 넓은 상권에 공유되는 구조가 확인된다. 이러한 구조는 실제 공연 소비와 추정 노출 사이의 불일치를 만들 수 있다. 다만 현재 결과가 비유의한 이유를 이 구조로 확정하거나, 자료를 바꾸면 유의해질 것이라고 예측할 수는 없다.